# Feature Engineering

Turning the raw columns from the EDA notebook into features a model can actually use - splitting combined strings, encoding Yes/No flags, one hot encoding categoricals and scaling numeric columns. The same logic lives in `src/data_transformation.py` so training and the Flask app stay in sync.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import re

from src.data_transformation import DataTransformation

df = pd.read_csv('../data/train.csv')
df.shape

(58592, 44)

### Step 1 - splitting max_torque and max_power

Both columns are strings like `113Nm@4400rpm`. A regex split gives two clean numeric columns each: the value and the rpm at which it is achieved.

In [2]:
sample = df['max_torque'].head()
print(sample.tolist())

torque_split = df['max_torque'].str.extract(r'([\d.]+)Nm@([\d.]+)rpm')
torque_split.columns = ['torque_nm', 'torque_rpm']
torque_split.head()

['60Nm@3500rpm', '60Nm@3500rpm', '60Nm@3500rpm', '113Nm@4400rpm', '91Nm@4250rpm']


,torque_nm,torque_rpm
0,60,3500
1,60,3500
2,60,3500
3,113,4400
4,91,4250


In [3]:
power_split = df['max_power'].str.extract(r'([\d.]+)bhp@([\d.]+)rpm')
power_split.columns = ['power_bhp', 'power_rpm']
power_split.head()

,power_bhp,power_rpm
0,40.36,6000
1,40.36,6000
2,40.36,6000
3,88.50,6000
4,67.06,5500


### Step 2 - encoding Yes/No flag columns

There are 17 boolean-style columns stored as `Yes`/`No` strings (airbags, parking sensors, brake assist and so on). Mapping them to 1/0 keeps them usable as numeric features without one hot encoding overhead.

In [4]:
yes_no_cols = [c for c in df.columns if df[c].dropna().isin(['Yes', 'No']).all() and df[c].nunique() == 2]
print('Yes/No columns found:', len(yes_no_cols))
yes_no_cols

Yes/No columns found: 17


['is_esc',
 'is_adjustable_steering',
 'is_tpms',
 'is_parking_sensors',
 'is_parking_camera',
 'is_front_fog_lights',
 'is_rear_window_wiper',
 'is_rear_window_washer',
 'is_rear_window_defogger',
 'is_brake_assist',
 'is_power_door_locks',
 'is_central_locking',
 'is_power_steering',
 'is_driver_seat_height_adjustable',
 'is_day_night_rear_view_mirror',
 'is_ecw',
 'is_speed_alert']

In [5]:
for col in yes_no_cols:
    print(col, '->', df[col].unique())

is_esc -> ['No' 'Yes']
is_adjustable_steering -> ['No' 'Yes']
is_tpms -> ['No' 'Yes']
is_parking_sensors -> ['Yes' 'No']
is_parking_camera -> ['No' 'Yes']
is_front_fog_lights -> ['No' 'Yes']
is_rear_window_wiper -> ['No' 'Yes']
is_rear_window_washer -> ['No' 'Yes']
is_rear_window_defogger -> ['No' 'Yes']
is_brake_assist -> ['No' 'Yes']
is_power_door_locks -> ['No' 'Yes']
is_central_locking -> ['No' 'Yes']
is_power_steering -> ['Yes' 'No']
is_driver_seat_height_adjustable -> ['No' 'Yes']
is_day_night_rear_view_mirror -> ['No' 'Yes']
is_ecw -> ['No' 'Yes']
is_speed_alert -> ['Yes' 'No']


### Step 3 - categorical columns

`area_cluster`, `segment`, `model`, `fuel_type`, `engine_type`, `rear_brakes_type`, `transmission_type` and `steering_type` are true categorical columns with no ordinal meaning, so they go through one hot encoding rather than label encoding.

In [6]:
cat_cols = ['area_cluster', 'segment', 'model', 'fuel_type',
            'engine_type', 'rear_brakes_type', 'transmission_type',
            'steering_type']
for col in cat_cols:
    print(col, '-> unique values:', df[col].nunique())

area_cluster -> unique values: 22
segment -> unique values: 6
model -> unique values: 11
fuel_type -> unique values: 3
engine_type -> unique values: 11
rear_brakes_type -> unique values: 2
transmission_type -> unique values: 2
steering_type -> unique values: 3


### Putting it together

All of the steps above are already implemented in `DataTransformation` inside `src/data_transformation.py` - running it here on the full dataset to see the final shape and confirm nothing breaks.

In [7]:
transformer = DataTransformation(artifacts_dir='../artifacts')
X, y = transformer.fit_transform(df)
print('Final feature matrix shape:', X.shape)
X.head()

2026-07-18 11:49:39,982 | src.data_transformation | INFO | Fitting data transformation pipeline
2026-07-18 11:49:40,614 | src.data_transformation | INFO | Final feature matrix shape: (58592, 96)


Final feature matrix shape: (58592, 96)


,policy_tenure,age_of_car,age_of_policyholder,population_density,make,airbags,is_esc,is_adjustable_steering,is_tpms,is_parking_sensors,...,engine_type_K Series Dual jet,engine_type_K10C,engine_type_i-DTEC,rear_brakes_type_Disc,rear_brakes_type_Drum,transmission_type_Automatic,transmission_type_Manual,steering_type_Electric,steering_type_Manual,steering_type_Power
0,-0.230283,-0.342447,1.422557,-0.783513,-0.671712,-0.620458,-0.676638,-1.241044,-0.560793,0.205451,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
1,0.148188,-0.871359,-0.768362,0.462975,-0.671712,-0.620458,-0.676638,-1.241044,-0.560793,0.205451,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,0.555022,-0.871359,-0.690115,-0.835268,-0.671712,-0.620458,-0.676638,-1.241044,-0.560793,0.205451,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,0.697883,0.715378,-0.298879,0.158275,-0.671712,-0.620458,1.477895,0.805774,-0.560793,0.205451,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
4,-0.035840,0.715378,1.344310,0.900969,0.207812,-0.620458,-0.676638,-1.241044,-0.560793,-4.867351,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0


In [8]:
print('Total engineered features:', X.shape[1])
print('Numeric columns used:', len(transformer.numeric_columns))
print('Target distribution after transform:')
y.value_counts(normalize=True).round(4)

Total engineered features: 96
Numeric columns used: 36
Target distribution after transform:


is_claim
0    0.936
1    0.064
Name: proportion, dtype: float64

## Summary

- `max_torque` / `max_power` split into 4 numeric columns.
- 17 Yes/No columns mapped to binary 1/0.
- 8 categorical columns one hot encoded.
- Remaining numeric columns scaled with `StandardScaler`.
- Fitted encoder and scaler saved to `artifacts/` so the same transformation can run on new data at prediction time.

Next notebook: `03_feature_selection.ipynb`